# LangChain Agent — 영업 메일 도우미

같은 시나리오로 만드는 **세 번째** 노트북입니다. 앞의 두 개와 비교하면
프레임워크가 무엇을 대신해 주는지가 분명해집니다.

| | 흐름을 정하는 주체 |
|---|---|
| OpenAI Agents SDK | 우리 코드 + 에이전트 |
| LangGraph | 우리가 그린 **그래프** |
| **LangChain Agent (이번)** | **에이전트가 스스로**, 우리는 **미들웨어로 가로채기만** |

## 시나리오

윈스테크넷에서 **스나이퍼 APT**(APT 대응 솔루션)를 파는 영업 담당자입니다.
고객사를 조회해 맞춤 메일을 쓰고 발송합니다.

## 이번에 배우는 여섯 가지

| 개념 | 한 줄 설명 | 단계 |
|---|---|---|
| `@tool` | 평범한 함수를 에이전트의 도구로 | Step 1 |
| `create_agent` | 도구를 가진 에이전트를 **한 줄로** | Step 2 |
| `ainvoke` | 여러 요청을 **동시에** | Step 4 |
| 구조화 출력 | 답을 **문자열이 아니라 객체로** | Step 5 |
| 메모리 | 지난 대화를 **기억** | Step 6 |
| 미들웨어 | 에이전트 루프 중간에 **끼어들기** | Step 7 |

> 실행 순서: 위에서부터 차례대로. `agent/` 폴더에서 실행하세요.

## Step 0. 준비

`langchain` 본체 패키지가 필요합니다(`langchain-core`만으로는 `create_agent`가 없습니다).

```bash
uv add langchain
```

In [1]:
import asyncio

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool

load_dotenv(override=True)

MODEL = "openai:gpt-5.4-mini"   # "제공사:모델명" 형식
print("모델:", MODEL)

모델: openai:gpt-5.4-mini


모델을 `"openai:gpt-5.4-mini"`처럼 **문자열 하나로** 지정합니다.
`ChatOpenAI` 객체를 만들 필요가 없고, 제공사를 바꾸려면 앞부분만 고치면 됩니다.

## Step 1. 도구 만들기

`@tool`을 붙이면 평범한 함수가 에이전트가 쓸 수 있는 도구가 됩니다.

모델은 **함수 본문을 보지 않습니다.** 이름, 타입 힌트, 독스트링만 보고
"언제 이 도구를 쓸지"를 판단합니다. 그래서 독스트링은 주석이 아니라 **명세서**입니다.

In [2]:
# 고객사 정보 (실제로는 CRM 데이터베이스)
CUSTOMERS = {
    "한빛테크": {"담당자": "김민수 팀장", "업종": "제조", "직원수": 1200},
    "가온소프트": {"담당자": "이서연 이사", "업종": "금융", "직원수": 300},
    "세움물산": {"담당자": "박지훈 과장", "업종": "유통", "직원수": 80},
}

sent_emails = []        # 발송 기록 (실제 발송 대신 여기에 쌓는다)


@tool
def find_customer(name: str) -> str:
    """고객사 이름으로 담당자와 규모를 조회한다.

    Args:
        name: 고객사 이름
    """
    info = CUSTOMERS.get(name)
    return f"{name} - {info}" if info else f"{name}: 등록되지 않은 고객사입니다."


@tool
def get_product_spec(topic: str) -> str:
    """스나이퍼 APT 제품 정보를 조회한다.

    Args:
        topic: 알고 싶은 항목 (기능 / 가격 / 도입사례 중 하나)
    """
    spec = {
        "기능": "알려지지 않은 악성코드와 표적 공격을 네트워크 단에서 탐지·차단",
        "가격": "연간 구독. 네트워크 구간 수에 따라 산정",
        "도입사례": "제조·금융권 중심 300여 곳",
    }
    return spec.get(topic, "해당 항목 정보가 없습니다.")


@tool
def send_email(subject: str, body: str) -> str:
    """영업 메일을 발송한다.

    Args:
        subject: 메일 제목
        body: 메일 본문
    """
    sent_emails.append(subject)
    return f"발송 완료: {subject}"


print("도구 3개 준비:", [t.name for t in [find_customer, get_product_spec, send_email]])

도구 3개 준비: ['find_customer', 'get_product_spec', 'send_email']


> 실제 발송 대신 목록에 쌓기만 합니다. 되돌릴 수 없는 동작은 학습 단계에서
> 꺼 두는 편이 안전합니다. Step 7에서 **미들웨어로 승인 절차**를 붙입니다.

## Step 2. `create_agent` — 에이전트를 한 줄로

```python
create_agent(model=..., tools=[...], system_prompt="...")
```

이 한 번의 호출이 **2.2절의 에이전트 루프**(판단 → 행동 → 관찰)를 통째로 만들어 줍니다.
우리가 루프를 돌리거나, 도구를 실행하거나, 결과를 모델에 되돌릴 필요가 없습니다.

In [3]:
SYSTEM = """당신은 윈스테크넷의 영업 담당자입니다.
'스나이퍼 APT'라는 APT 대응 솔루션을 판매합니다.
고객사 정보나 제품 정보가 필요하면 반드시 도구로 확인하세요. 추측하지 마세요.
한국어로 간결하게 답하세요."""

agent = create_agent(
    model=MODEL,
    tools=[find_customer, get_product_spec],
    system_prompt=SYSTEM,
)

print("에이전트 준비 완료")

에이전트 준비 완료


## Step 3. 실행하고 속을 들여다보기

입력은 **메시지 목록**이고, 결과도 메시지 목록입니다.

```python
agent.invoke({"messages": [{"role": "user", "content": "..."}]})
```

In [4]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "한빛테크 담당자가 누구야?"}]
})

print(result["messages"][-1].content)

한빛테크 담당자는 **김민수 팀장**입니다.


### 에이전트가 거쳐 간 길

최종 답변만 보면 모델이 그냥 안 것처럼 보입니다. **메시지 목록을 펼쳐 보면**
도구를 부르고 결과를 받아 다시 판단한 과정이 그대로 남아 있습니다.

In [5]:
for m in result["messages"]:
    kind = type(m).__name__
    if getattr(m, "tool_calls", None):
        print(f"{kind:14} 도구 호출 -> {[c['name'] for c in m.tool_calls]}")
    else:
        print(f"{kind:14} {str(m.content)[:70]}")

HumanMessage   한빛테크 담당자가 누구야?
AIMessage      도구 호출 -> ['find_customer']
ToolMessage    한빛테크 - {'담당자': '김민수 팀장', '업종': '제조', '직원수': 1200}
AIMessage      한빛테크 담당자는 **김민수 팀장**입니다.


```
HumanMessage   한빛테크 담당자가 누구야?        <- 질문
AIMessage      도구 호출 -> ['find_customer']   <- 판단
ToolMessage    한빛테크 - {'담당자': ...}        <- 행동의 결과(관찰)
AIMessage      한빛테크 담당자는 김민수 팀장입니다  <- 최종 판단
```

**이 네 줄이 에이전트 루프입니다.** LangGraph에서는 노드와 엣지로 직접 그렸고,
여기서는 `create_agent`가 대신 만들어 줍니다.

## Step 4. `ainvoke` — 여러 요청을 동시에

고객사 세 곳을 조사해야 한다면, 서로 기다릴 이유가 없습니다.

| | 하는 일 |
|---|---|
| `invoke` | 하나를 끝까지 처리하고 돌려줌 |
| `ainvoke` | **기다리는 동안 다른 일을 할 수 있는** 형태로 처리 |

`ainvoke`는 `await`과 함께 쓰고, `asyncio.gather`로 묶으면 동시에 돌아갑니다.
LLM 호출은 대부분의 시간을 **응답을 기다리는 데** 쓰기 때문에 이득이 큽니다.

In [6]:
import time

questions = [f"{name} 담당자와 업종을 알려줘" for name in CUSTOMERS]


async def run_all():
    return await asyncio.gather(*(
        agent.ainvoke({"messages": [{"role": "user", "content": q}]}) for q in questions
    ))


start = time.perf_counter()
results = await run_all()
print(f"3건 동시 처리: {time.perf_counter() - start:.1f}초\n")

for r in results:
    print(" -", r["messages"][-1].content[:60])

3건 동시 처리: 1.7초

 - 한빛테크 담당자는 **김민수 팀장**이고, 업종은 **제조**입니다.
 - 가온소프트 담당자는 **이서연 이사**이고, 업종은 **금융**입니다.
 - 세움물산 담당자는 **박지훈 과장**이고, 업종은 **유통**입니다.


## Step 5. 구조화 출력 — 문자열 대신 객체로

지금까지 답은 **사람이 읽는 글**이었습니다. 그런데 메일 초안을 발송 시스템에
넘기려면 제목과 본문이 **따로** 필요합니다. 글에서 뽑아내려면 파싱 코드가 생깁니다.

`response_format`에 원하는 모양을 주면 모델이 **그 모양으로** 답합니다.

In [7]:
from pydantic import BaseModel, Field


# 클래스 이름과 칸 이름은 영문으로. 한글 설명은 description 에 넣는다 (아래 주의)
class EmailDraft(BaseModel):
    """영업 메일 초안."""
    subject: str = Field(description="메일 제목")
    body: str = Field(description="메일 본문. 3~4문장.")
    recipient: str = Field(description="받는 사람 이름과 직함")


draft_agent = create_agent(
    model=MODEL,
    tools=[find_customer, get_product_spec],
    system_prompt=SYSTEM,
    response_format=EmailDraft,        # <- 이 한 줄
)

out = draft_agent.invoke({
    "messages": [{"role": "user", "content": "한빛테크에 보낼 영업 메일 초안을 써줘"}]
})

draft = out["structured_response"]       # <- 여기에 담겨 온다
print(type(draft).__name__)
print("제목  :", draft.subject)
print("수신자:", draft.recipient)
print("본문  :", draft.body[:100], "...")

EmailDraft
제목  : 한빛테크 김민수 팀장님께, 네트워크 기반 APT 대응 제안드립니다
수신자: 김민수 팀장
본문  : 안녕하세요, 김민수 팀장님. 윈스테크넷입니다. 한빛테크와 같은 제조 환경에서는 알려지지 않은 악성코드와 표적 공격 대응이 중요해 스나이퍼 APT를 제안드립니다. 스나이퍼 APT는  ...


**이제 파싱이 필요 없습니다.** `초안.subject`를 그대로 발송 함수에 넘기면 됩니다.

`Field(description=...)`는 모델에게 "이 칸에 무엇을 넣어야 하는지" 알려 줍니다.
설명을 비워 두면 모델이 칸의 의도를 짐작해야 하므로 엉뚱한 값이 들어갑니다.

> **주의 — 이름에는 한글을 쓸 수 없습니다.** 클래스 이름을 `메일초안`으로 두면
> 이런 오류가 납니다 (실제로 겪은 것입니다).
>
> ```
> Invalid 'response_format.json_schema.name': string does not match pattern.
> Expected a string that matches the pattern '^[a-zA-Z0-9_-]+$'.
> ```
>
> 클래스 이름이 그대로 JSON 스키마 이름으로 전달되기 때문입니다.
> 도구 이름(`@tool`)에도 같은 규칙이 적용됩니다.
> **이름은 영문, 설명은 한글** — 모델은 이름보다 설명을 보고 판단하므로
> 품질에는 영향이 없습니다.

In [8]:
# 구조화 출력은 '검증된' 객체다. 타입이 맞지 않으면 애초에 만들어지지 않는다.
print(draft.model_dump_json(indent=2)[:300])

{
  "subject": "한빛테크 김민수 팀장님께, 네트워크 기반 APT 대응 제안드립니다",
  "body": "안녕하세요, 김민수 팀장님. 윈스테크넷입니다. 한빛테크와 같은 제조 환경에서는 알려지지 않은 악성코드와 표적 공격 대응이 중요해 스나이퍼 APT를 제안드립니다. 스나이퍼 APT는 네트워크 단에서 알려지지 않은 악성코드와 표적 공격을 탐지·차단해 보안 운영 부담을 줄이는 데 도움을 드립니다. 검토 가능하시면 짧게 소개드릴 수 있도록 미팅 일정 조율 부탁드립니다.",
  "recipient": "김민수 팀장"
}


## Step 6. 메모리 — 지난 대화를 기억하기

지금까지의 에이전트는 **매번 처음부터**였습니다. "그 회사"라고 물으면 모릅니다.

`checkpointer`를 주면 대화 상태가 저장되고, `thread_id`로 이어서 부를 수 있습니다.

```python
create_agent(..., checkpointer=InMemorySaver())
agent.invoke(..., {"configurable": {"thread_id": "고객사-A"}})
```

`thread_id`는 **대화를 구분하는 이름표**입니다. 같은 id면 이어지고, 다르면 별개입니다.

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(
    model=MODEL,
    tools=[find_customer, get_product_spec],
    system_prompt=SYSTEM,
    checkpointer=InMemorySaver(),      # <- 이 한 줄
)

config = {"configurable": {"thread_id": "한빛테크-상담"}}

a = memory_agent.invoke({"messages": [{"role": "user", "content": "한빛테크 담당자 누구야?"}]}, config)
print("1턴:", a["messages"][-1].content[:70])

b = memory_agent.invoke({"messages": [{"role": "user", "content": "그 회사 직원수는?"}]}, config)
print("2턴:", b["messages"][-1].content[:70])

1턴: 한빛테크 담당자는 **김민수 팀장**입니다.
2턴: 한빛테크 직원수는 **1,200명**입니다.


2턴에서 **"그 회사"가 한빛테크임을 알고 있습니다.** 1턴의 대화가 남아 있기 때문입니다.

`thread_id`를 바꾸면 어떻게 될까요?

In [10]:
other_config = {"configurable": {"thread_id": "가온소프트-상담"}}
c = memory_agent.invoke({"messages": [{"role": "user", "content": "그 회사 직원수는?"}]}, other_config)
print("다른 대화:", c["messages"][-1].content[:90])

다른 대화: 어느 회사를 말씀하시는지 회사명을 알려주시면 확인해드리겠습니다.


> `InMemorySaver`는 **메모리에만** 저장합니다. 커널을 다시 시작하면 사라집니다.
> 실제 서비스에서는 SQLite나 Postgres 체크포인터를 씁니다.

## Step 7. 미들웨어 — 루프 중간에 끼어들기

여기가 LangChain Agent의 핵심입니다.

`create_agent`는 편한 대신 **속이 보이지 않습니다.** 모델이 도구를 부르면 그냥 실행됩니다.
그런데 메일 발송처럼 **되돌릴 수 없는 동작**은 사람이 한 번 봐야 합니다.

미들웨어는 에이전트 루프의 정해진 지점에 **우리 코드를 끼워 넣는** 장치입니다.
직접 만들 수도 있지만, 자주 쓰는 것들은 이미 만들어져 있습니다.

In [11]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

require_approval = HumanInTheLoopMiddleware(interrupt_on={"send_email": True})

sender_agent = create_agent(
    model=MODEL,
    tools=[find_customer, get_product_spec, send_email],
    system_prompt=SYSTEM + "\n메일을 다 쓰면 send_email 도구로 발송하세요.",
    middleware=[require_approval],            # <- 이 한 줄
    checkpointer=InMemorySaver(),     # 중단·재개하려면 체크포인터가 필요하다
)

print("발송 에이전트 준비 (send_email 은 승인 필요)")

발송 에이전트 준비 (send_email 은 승인 필요)


### 실행하면 중간에 멈춥니다

In [12]:
config1 = {"configurable": {"thread_id": "승인-예시"}}

result = sender_agent.invoke(
    {"messages": [{"role": "user", "content": "한빛테크에 짧은 인사 메일 보내줘"}]},
    config1,
)

print("중단됨:", "__interrupt__" in result)
request = result["__interrupt__"][0].value["action_requests"][0]
print("승인 요청 도구:", request["name"])
print("제목:", request["args"]["subject"])
print()
print("지금까지 발송된 메일:", sent_emails)

중단됨: True
승인 요청 도구: send_email
제목: 한빛테크 김민수 팀장님께 인사드립니다

지금까지 발송된 메일: []


**아직 메일은 나가지 않았습니다.** 에이전트가 발송 직전에 멈춰 서서 우리에게 물었습니다.

이제 사람이 결정을 돌려주면 이어서 진행합니다.

| 결정 | 뜻 |
|---|---|
| `approve` | 그대로 실행 |
| `reject` | 실행하지 않음 (이유를 함께 전달 가능) |
| `edit` | 인자를 고쳐서 실행 |
| `respond` | 도구 대신 사람이 답을 줌 |

In [13]:
final = sender_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config1,
)

print("최종 답변:", final["messages"][-1].content[:80])
print("발송된 메일:", sent_emails)

최종 답변: 발송했습니다.
발송된 메일: ['한빛테크 김민수 팀장님께 인사드립니다']


### 거부하면 어떻게 되나

In [14]:
config2 = {"configurable": {"thread_id": "거부-예시"}}

sender_agent.invoke(
    {"messages": [{"role": "user", "content": "세움물산에 영업 메일 보내줘"}]},
    config2,
)
rejected = sender_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "표현이 과합니다. 다시 검토하세요."}]}),
    config2,
)

print("도구 결과:", [m.content[:80] for m in rejected["messages"] if type(m).__name__ == "ToolMessage"][-1:])
print("발송된 메일:", sent_emails, "   <- 늘지 않았다")

도구 결과: ['User rejected the tool call for `send_email` with reason: 표현이 과합니다. 다시 검토하세요.']
발송된 메일: ['한빛테크 김민수 팀장님께 인사드립니다']    <- 늘지 않았다


### 이미 만들어진 미들웨어들

승인 외에도 자주 쓰는 것들이 준비돼 있습니다. 모두 `middleware=[...]`에 넣기만 하면 됩니다.

| 미들웨어 | 하는 일 |
|---|---|
| `HumanInTheLoopMiddleware` | 위험한 도구 실행 전 사람 승인 |
| `PIIMiddleware` | 개인정보(전화번호·이메일 등)를 모델에 넘기기 전에 가림 |
| `ModelCallLimitMiddleware` | 한 번의 실행에서 모델 호출 횟수 제한 (폭주 방지) |
| `ModelFallbackMiddleware` | 모델이 실패하면 다른 모델로 |
| `ModelRetryMiddleware` | 일시적 오류 시 재시도 |
| `LLMToolSelectorMiddleware` | 도구가 많을 때 후보를 먼저 추려 줌 |

> 미들웨어가 중요한 이유: `create_agent`는 **편한 대신 안이 가려집니다.**
> 미들웨어는 그 안으로 들어가는 **정해진 문**입니다. 문을 열지 않으면
> 결국 LangGraph로 내려가 직접 그려야 합니다.

## Step 8. 정리 — 세 프레임워크

같은 영업 메일 시나리오를 세 가지로 만들어 봤습니다.

| | OpenAI Agents SDK | LangGraph | LangChain Agent |
|---|---|---|---|
| 만드는 법 | `Agent(...)` | 노드·엣지를 직접 | **`create_agent(...)`** |
| 코드 양 | 적음 | **많음** | **가장 적음** |
| 흐름이 보이나 | 보통 | **전부 보임** | **가려짐** |
| 되돌아가는 고리 | 어려움 | **쉬움** | 어려움 |
| 중간에 끼어들기 | 도구 함수 안에서 | 노드 추가 | **미들웨어** |
| 기억 | `SQLiteSession` | 체크포인터 | 체크포인터 |
| 구조화 출력 | `output_type` | 직접 | `response_format` |

### 무엇을 고를까

- **빨리 만들고 싶다** → LangChain Agent. 여섯 줄이면 도구 쓰는 에이전트가 됩니다.
- **흐름을 눈으로 봐야 한다** → LangGraph. 그래프가 곧 설계도입니다.
- **되돌아가는 고리가 있다** → LangGraph. 다른 선택지가 없습니다.
- **표준 기능(승인·PII·재시도)만 필요하다** → LangChain Agent + 미들웨어.

> `create_agent`는 내부적으로 **LangGraph 그래프를 만들어 줍니다.**
> 즉 둘은 경쟁 관계가 아니라 층위가 다릅니다. 기성품으로 시작하고,
> 모자라면 한 층 내려가면 됩니다.

### 이 노트북에서 기억할 것

| | |
|---|---|
| 독스트링은 명세서 | 모델은 함수 본문을 보지 않습니다 |
| 메시지 목록을 펼쳐 보기 | 최종 답변만 보면 루프가 안 보입니다 |
| 구조화 출력 | 파싱 코드가 통째로 사라집니다 |
| `thread_id` | 대화를 구분하는 이름표 |
| 되돌릴 수 없는 동작엔 승인 | 미들웨어 한 줄 |

### 더 해볼 것

1. `interrupt_on`에 `get_product_spec`도 넣어 보세요. 조회에도 승인을 요구하면 얼마나 번거로운지 보입니다.
2. 거부할 때 `{"type": "edit", ...}`로 제목을 고쳐 보세요.
3. `PIIMiddleware`를 추가하고 고객 메일 주소가 포함된 질문을 던져 보세요.
4. 도구를 하나 더 만들어 `tools=[...]`에 추가해 보세요. 그 외에는 바꿀 것이 없습니다.